In [ ]:
import mdtraj as md
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# --- Load Trajectory ---
traj = md.load("rfc.dcd", top="rfc.pdb")

# --- Select protein backbone atoms ---
bb = traj.topology.select("protein and backbone")

# --- Align on first frame ---
traj.superpose(traj, frame=0, atom_indices=bb)

# --- Build matrix for PCA --- 
# Shape: (n_frames, n_atoms*3)
X = traj.xyz[:, bb, :].reshape(traj.n_frames, -1)


# --- Run PCA (sklearn centers data by default) ---
pca = PCA(n_components=5)       # grab first few PCs
proj = pca.fit_transform(X)     # projections onto PCs
PCs = pca.components_           # shape: (n_components, n_features)
X_mean = pca.mean_              # mean structure

In [ ]:
# --- Scatter plot PC1 vs PC2 ---
plt.figure(figsize=(6, 5))
plt.scatter(
    proj[:, 0],
    proj[:, 1],
    c=np.arange(traj.n_frames),
    cmap="viridis",
    s=10
)
plt.colorbar(label="Frame")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Backbone PCA (MDTraj + scikit-learn)")
plt.tight_layout()
plt.show()

In [ ]:
# --- Plot PC1 over time (Frame Index) ---
plt.figure(figsize=(8, 4))
plt.plot(proj[:, 0], color='purple')
plt.xlabel('Frame Number')
plt.ylabel('PC1 Projection')
plt.title('PC1 Evolution over Time')
plt.show()

In [ ]:
# --- Plot Explained Variance ---
plt.figure(figsize=(6, 4))
var_exp = pca.explained_variance_ratio_ * 100
cum_var_exp = np.cumsum(var_exp)

plt.bar(range(1, 6), var_exp, alpha=0.5, align='center', label='Individual')
plt.step(range(1, 6), cum_var_exp, where='mid', label='Cumulative', color='red')
plt.ylabel('Percentage of Variance Explained')
plt.xlabel('Principal Component Index')
plt.title('Scree Plot')
plt.legend()
plt.show()

In [ ]:
# --- Calculate per-atom contribution to PC1 ---
# Square the components to get 'importance' and sum the x,y,z contributions per atom
pc1_contributions = np.sum(np.square(PCs[0].reshape(-1, 3)), axis=1)

# Plotting by atom index
plt.figure(figsize=(8, 4))
plt.plot(pc1_contributions, color='darkblue')
plt.xlabel('Atom Index (Backbone)')
plt.ylabel('Contribution to PC1')
plt.title('Which atoms drive the primary motion?')
plt.show()

In [ ]:
# --- Make morph movies along PC1 and PC2 ---
def write_pc_morph_clean(pc_index, n_steps=30, n_std=2.0, outname="pc_morph_selected.pdb"):
    """
    Writes a morph movie containing ONLY the atoms used in the PCA.
    """
    # --- Get PCA components for the selection ---
    pc_vec = PCs[pc_index].reshape(-1, 3)
    mean_xyz = X_mean.reshape(-1, 3)
    
    # --- make the scale ---
    std_dev = np.sqrt(pca.explained_variance_[pc_index])
    scale = std_dev * n_std

    # --- Generate the interpolated frames for the selection ---
    morph_xyz = []
    for alpha in np.linspace(-scale, scale, n_steps):
        # We start with the mean structure of the selection and add the PC vector
        frame_xyz = mean_xyz + (alpha * pc_vec)
        morph_xyz.append(frame_xyz)
    
    # ---- Slice the original topology to match the selection (e.g., backbone) ---
    # This removes waters, ions, and other protein atoms not in 'bb'
    selected_topo = traj.topology.subset(bb)

    # --- Create the new trajectory with the reduced topology ---
    # np.array(morph_xyz) shape: (n_steps, n_selected_atoms, 3)
    morph = md.Trajectory(np.array(morph_xyz), selected_topo)
    
    # --- Save ---
    morph.save_pdb(outname)
    print(f"Success: Wrote {len(bb)} atoms to {outname}")

# Run for PC1
write_pc_morph_clean(0, outname="pc1_backbone_only.pdb")
write_pc_morph_clean(1, outname="pc2_backbone_only.pdb")

### **Homework for the reader**
How do the residues that contribute most to PC1 correlate to the RMSF values? What would you expect?